This notebook demonstrates the process of fine-tuning a GOT-OCR2.0 model for Telugu Handwritten Text Recognition and evaluating its performance.

In [1]:
!nvidia-smi

Wed Jul  8 04:30:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Check GPU Status
This cell runs the `nvidia-smi` command to display information about the GPU, including its driver version, CUDA version, and current memory usage. This is important to ensure that a GPU is available and properly configured for training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Mount Google Drive
This cell mounts your Google Drive to the Colab environment. This allows the notebook to access files stored in your Google Drive, such as datasets, checkpoints, and output directories.

In [ ]:
!ls /content/drive/MyDrive

'Colab Notebooks'	    IIIT-HW-Telugu.zip	 Train_full.jsonl
'Copy of test_full.jsonl'   Telugu_GOT_OCR	 Val_2k.jsonl


### List Google Drive Contents
This command lists the contents of your Google Drive's root directory (`/content/drive/MyDrive`). This helps in verifying that the necessary files and folders (like the dataset zip file or project directory) are accessible.

In [ ]:

!cp "/content/drive/MyDrive/IIIT-HW-Telugu.zip" /content/IIIT-HW-Telugu.zip

!unzip -q /content/IIIT-HW-Telugu.zip -d /content/


!unzip -q -o /content/IIIT-HW-Telugu/TeluguSeg.zip -d /content/IIIT-HW-Telugu/

### Copy and Unzip Dataset
This cell first copies the `IIIT-HW-Telugu.zip` dataset from Google Drive to the Colab environment's content directory. Then, it unzips the main dataset, and subsequently unzips the `TeluguSeg.zip` file located within the extracted `IIIT-HW-Telugu` directory. This prepares the image and text data for use.

In [ ]:
!ls /content/IIIT-HW-Telugu

lexicon.txt  TeluguSeg	    telugu_vocab.txt  train.txt
Readme.txt   TeluguSeg.zip  test.txt	      val.txt


### List Unzipped Dataset Contents
This command lists the contents of the newly unzipped `IIIT-HW-Telugu` directory, showing the structure of the dataset files (e.g., `lexicon.txt`, `train.txt`, `val.txt`, `test.txt`, `TeluguSeg`).

In [ ]:
with open('/content/IIIT-HW-Telugu/train.txt','r',encoding='utf-8') as f:
    for i in range(10):
        print(f.readline())

TeluguSeg/train/3/259/1.jpg పర్వతాన్ని

TeluguSeg/train/3/259/2.jpg కలుషితమై

TeluguSeg/train/3/259/3.jpg కంట్రాక్టర్లు

TeluguSeg/train/3/259/4.jpg తీసుకోబోతున్నాడని

TeluguSeg/train/3/259/5.jpg సంరక్షకులుగా

TeluguSeg/train/3/259/6.jpg కవర్లు

TeluguSeg/train/3/259/7.jpg తాకడం

TeluguSeg/train/3/259/9.jpg పదుకొనె

TeluguSeg/train/3/259/10.jpg చెప్పాయి

TeluguSeg/train/3/259/11.jpg ప్రతివ్యూహాలు



### Preview `train.txt`
This cell opens the `train.txt` file and prints the first 10 lines. This helps in understanding the format of the training data, which typically consists of an image path followed by its corresponding text label.

In [ ]:
import os

img = "/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg"

print(os.path.exists(img))

True


### Verify Image Path
This cell checks if a specific image file path exists. It constructs an absolute path to a sample image mentioned in `train.txt` and uses `os.path.exists()` to confirm its presence, which is crucial for image loading during training.

In [ ]:
def count_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

print("Train:", count_lines('/content/IIIT-HW-Telugu/train.txt'))
print("Val:", count_lines('/content/IIIT-HW-Telugu/val.txt'))
print("Test:", count_lines('/content/IIIT-HW-Telugu/test.txt'))

Train: 80637
Val: 19980
Test: 17898


### Count Lines in Dataset Files
This cell defines a helper function `count_lines` to count the number of lines in a given text file. It then uses this function to display the total number of samples in the `train.txt`, `val.txt`, and `test.txt` files, providing an overview of the dataset sizes.

In [ ]:
with open('/content/IIIT-HW-Telugu/train.txt', 'r', encoding='utf-8') as f:
    for i in range(5):
        print(f.readline().strip())

TeluguSeg/train/3/259/1.jpg పర్వతాన్ని
TeluguSeg/train/3/259/2.jpg కలుషితమై
TeluguSeg/train/3/259/3.jpg కంట్రాక్టర్లు
TeluguSeg/train/3/259/4.jpg తీసుకోబోతున్నాడని
TeluguSeg/train/3/259/5.jpg సంరక్షకులుగా


### Preview `train.txt` (Stripped)
This cell again previews the first 5 lines of the `train.txt` file, but this time it uses `.strip()` to remove any leading/trailing whitespace or newline characters. This gives a cleaner view of the actual image path and text label.

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Telugu_GOT_OCR"

JSON_DIR = os.path.join(PROJECT_DIR, "jsons")

os.makedirs(JSON_DIR, exist_ok=True)

print(JSON_DIR)

/content/drive/MyDrive/Telugu_GOT_OCR/jsons


### Setup Project and JSON Directory
This cell sets up the `PROJECT_DIR` in Google Drive and creates a `jsons` subdirectory within it. This directory will be used to store the processed JSONL dataset files, ensuring that the generated data is saved persistently.

In [ ]:
import json
import os

def create_swift_got_json(txt_file, output_json, max_samples=None):
    data = []

    with open(txt_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if max_samples and idx >= max_samples:
                break

            line = line.strip()
            if not line:
                continue

            # Split the line by the first whitespace into image path and the text label
            image_rel, text = line.split(maxsplit=1)

            # Form the absolute image path exactly how you did before
            image_abs = os.path.join("/content/IIIT-HW-Telugu", image_rel)

            # Format explicitly for MS-SWIFT's multi-modal structure
            # The <image> tag acts as the token positioning placeholder for the model
            sample = {
                "query": "<image>\nOCR:",
                "response": text.strip(),
                "images": [image_abs]
            }
            data.append(sample)

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

    print(f"Saved {len(data)} samples -> {output_json}")


create_swift_got_json(
    "/content/IIIT-HW-Telugu/train.txt",
    f"{JSON_DIR}/train_small.json",
    max_samples=5000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/val.txt",
    f"{JSON_DIR}/val_small.json",
    max_samples=1000
)

create_swift_got_json(
    "/content/IIIT-HW-Telugu/test.txt",
    f"{JSON_DIR}/test_small.json",
    max_samples=1000
)

Saved 5000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.json
Saved 1000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.json
Saved 1000 samples -> /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json


### Convert Text Files to MS-SWIFT JSON Format
This cell defines a function `create_swift_got_json` that reads a text file (like `train.txt`), parses each line into an image path and a text label, and then converts it into a specific JSON format required by the MS-SWIFT framework. It then applies this function to create `train_small.json`, `val_small.json`, and `test_small.json` files with a limited number of samples.

In [ ]:
!ls "/content/drive/MyDrive/Telugu_GOT_OCR/jsons"

test_100.jsonl	 test_small.jsonl  train_small.jsonl  val_small.jsonl
test_small.json  train_small.json  val_small.json


### List Generated JSON Files
This command lists the files within the `jsons` directory in Google Drive, confirming that the `train_small.json`, `val_small.json`, and `test_small.json` files (along with any previously created files) have been successfully generated.

In [ ]:
import json

with open(
    f"{JSON_DIR}/train_small.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(data[0])

{'query': '<image>\nOCR:', 'response': 'పర్వతాన్ని', 'images': ['/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg']}


### Load and Preview First JSON Sample
This cell loads the `train_small.json` file and prints the first entry. This helps in verifying that the JSON conversion was successful and that the data is in the expected format, including the `query`, `response`, and `images` fields.

In [ ]:
import json

def json_to_jsonl(json_file, jsonl_file):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    with open(jsonl_file, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saved {len(data)} samples to {jsonl_file}")

### Define JSON to JSONL Conversion Function
This cell defines a utility function `json_to_jsonl` that takes a JSON file and converts it into a JSON Lines (JSONL) format. In JSONL, each line is a separate JSON object, which is a common format for large datasets used in machine learning.

In [ ]:
json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl"
)

json_to_jsonl(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.json",
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
)

Saved 5000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl
Saved 1000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl
Saved 1000 samples to /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl


### Convert JSON Files to JSONL Format
This cell applies the `json_to_jsonl` function to convert the `train_small.json`, `val_small.json`, and `test_small.json` files into their respective JSONL formats (`train_small.jsonl`, `val_small.jsonl`, `test_small.jsonl`). This prepares the data for training with frameworks that prefer the JSONL format.

In [ ]:
with open(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl",
    "r",
    encoding="utf-8"
) as f:

    for i in range(3):
        print(f.readline())

{"query": "<image>\nOCR:", "response": "పర్వతాన్ని", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg"]}

{"query": "<image>\nOCR:", "response": "కలుషితమై", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/2.jpg"]}

{"query": "<image>\nOCR:", "response": "కంట్రాక్టర్లు", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/3.jpg"]}



### Preview First 3 JSONL Samples
This cell opens one of the newly created JSONL files (`train_small.jsonl`) and prints the first three lines. This allows for a quick verification of the JSONL format, ensuring each line is a valid JSON object.

In [ ]:
with open(
    "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl",
    "r",
    encoding="utf-8"
) as f:
    print(f.readline())

{"query": "<image>\nOCR:", "response": "పర్వతాన్ని", "images": ["/content/IIIT-HW-Telugu/TeluguSeg/train/3/259/1.jpg"]}



### Preview First JSONL Sample
This cell displays only the first line of the `train_small.jsonl` file. This is another quick check to confirm the structure of individual JSON objects within the JSONL file.

In [ ]:
# 1. Install the MS-SWIFT wrapper framework
!pip install "ms-swift[llm]" -U

# 2. Install supporting image decoding and training orchestration packages
!pip install timm transformers accelerate decord einops

### Install MS-SWIFT and Dependencies
This cell installs the `ms-swift` library with `llm` support, which is a wrapper framework for large language models, along with other crucial packages like `timm`, `transformers`, `accelerate`, `decord`, and `einops` that are required for model training and image processing.

In [ ]:
!pip install decord

### Install `decord`
This cell explicitly installs the `decord` library, which is used for efficient video and image decoding. It is listed separately, though it might have been installed as part of the previous cell's dependencies.

In [ ]:
!pip install verovio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 87.6 MB/s eta 0:00:00


### Install `verovio`
This cell installs the `verovio` library. While not directly visible in the immediate context of this notebook's core task (OCR), it might be a dependency for other components or used for specific visualization or musical notation processing tasks if the OCR model were to be applied to such domains.

In [ ]:
!pip install "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.2 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### Install `torchao`
This cell installs a specific version of the `torchao` library. `torchao` is an optimization library for PyTorch, which can provide performance enhancements for training and inference, particularly with features like mixed-precision training or quantization.

In [ ]:
!pip install "datasets==2.21.0"

### Install `datasets`
This cell installs a specific version (`2.21.0`) of the `datasets` library from Hugging Face. This library is essential for easily loading, processing, and sharing datasets in a standardized format, commonly used in machine learning workflows.

In [ ]:
import datasets
print("datasets version:", datasets.__version__)

datasets version: 2.21.0


### Check `datasets` Version
This cell imports the `datasets` library and prints its installed version. This is a good practice to ensure that the correct version of the library is being used, as compatibility issues can arise with different versions.

##Testing checkpoint-939 and analyzing the evaluation metrics and accuracy

In [ ]:
import os

CHECKPOINT = "/content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v0-20260619-045844/checkpoint-939"

print(os.path.exists(CHECKPOINT))
print(os.listdir(CHECKPOINT))

True
['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'additional_config.json', 'training_args.bin', 'args.json', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


### Verify Checkpoint Path and Contents
This cell defines the path to a model checkpoint and then verifies its existence using `os.path.exists()`. It also lists the contents of the checkpoint directory using `os.listdir()`, which helps confirm that all necessary model files and training artifacts are present.

In [ ]:
TEST_FILE = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"

print(os.path.exists(TEST_FILE))

True


### Verify Test File Path
This cell defines the path to the `test_small.jsonl` file and verifies its existence using `os.path.exists()`. This ensures that the dataset for inference is correctly located.

In [ ]:
import json

input_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl"
output_file = "/content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl"

with open(input_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

with open(output_file, "w", encoding="utf-8") as f:
    f.writelines(lines[:100])

print("Saved", len(lines[:100]), "samples")

Saved 100 samples


### Create a 100-Sample Test File
This cell reads the `test_small.jsonl` file and creates a new file, `test_100.jsonl`, containing only the first 100 samples. This smaller test set is often used for quick evaluations or debugging during development.

## Training 5000 images dataset with changed or revised parameters

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 100 \
    --save_strategy steps \
    --save_steps 100 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/train_small.jsonl --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/val_small.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 100 --save_strategy steps --save_steps 100 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not lo

### Train the Model (Small Dataset)
This cell initiates the training of the GOT-OCR2.0 model using the `swift sft` command. It specifies the base model, the `train_small.jsonl` and `val_small.jsonl` datasets, and various training parameters like output directory, mixed-precision settings, number of epochs, batch size, learning rate, and evaluation/save strategies. This training uses a smaller subset of the data for faster iteration.

## Evaluation of checkpoint-939

In [ ]:
pip install transformers==4.51.3

### Install Specific Transformers Version
This cell installs a precise version of the `transformers` library (4.51.3). Specifying the version helps ensure reproducibility and compatibility with the `swift` framework and the GOT model, as behavior can sometimes change between library versions.

In [ ]:
import glob

paths = glob.glob("/root/.cache/huggingface/modules/transformers_modules/**/modeling_GOT.py", recursive=True) + \
        glob.glob("/root/.cache/modelscope/hub/models/**/modeling_GOT.py", recursive=True)

print("Found:", paths)

for path in paths:
    with open(path, "r") as f:
        text = f.read()

    changed = False

    if "past_length = past_key_values.seen_tokens" in text:
        text = text.replace(
            "past_length = past_key_values.seen_tokens",
            "past_length = cache_length"
        )
        changed = True

    if "get_max_length()" in text:
        text = text.replace("get_max_length()", "get_seq_length()")
        changed = True

    if changed:
        with open(path, "w") as f:
            f.write(text)
        print(f"✅ Patched: {path}")
    else:
        print(f"Already patched or no match: {path}")

Found: ['/root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py', '/root/.cache/modelscope/hub/models/stepfun-ai/GOT-OCR2_0/modeling_GOT.py']
✅ Patched: /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py
✅ Patched: /root/.cache/modelscope/hub/models/stepfun-ai/GOT-OCR2_0/modeling_GOT.py


### Patch `modeling_GOT.py`
This cell finds instances of `modeling_GOT.py` within the Hugging Face and ModelScope caches and applies patches to specific lines of code. These patches likely address compatibility issues or bugs related to how `past_key_values` and sequence lengths are handled in the model's attention mechanism, which is crucial for correct inference.

In [ ]:
!grep -n "cache_length" /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py

399:                cache_length = past_key_values.get_seq_length()
400:                past_length = cache_length
401:                max_cache_length = past_key_values.get_seq_length()
403:                cache_length = past_length = past_key_values[0][0].shape[2]
404:                max_cache_length = None
420:                max_cache_length is not None
422:                and cache_length + input_ids.shape[1] > max_cache_length
424:                attention_mask = attention_mask[:, -max_cache_length:]


### Verify `cache_length` Patch
This command uses `grep` to search for the string "cache_length" within the patched `modeling_GOT.py` file and displays the lines where it appears. This is a verification step to confirm that the previous patching operation was successful and the relevant code lines have been modified as expected.

In [ ]:
!grep -n "get_seq_length" /root/.cache/huggingface/modules/transformers_modules/GOT-OCR2_0/modeling_GOT.py

399:                cache_length = past_key_values.get_seq_length()
401:                max_cache_length = past_key_values.get_seq_length()


### Verify `get_seq_length` Patch
Similar to the previous cell, this command uses `grep` to search for "get_seq_length" in the patched `modeling_GOT.py` file. This verifies that the replacement of `get_max_length()` with `get_seq_length()` was correctly applied, which is often necessary for models that handle variable sequence lengths efficiently.

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v0-20260619-045844/checkpoint-939 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v0-20260619-045844/checkpoint-939 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_100.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-19 08:28:59.104792: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical o

### Run Inference on 100-Sample Test Set (Checkpoint 939)
This cell executes an inference run using the `swift infer` command. It uses the model checkpoint-939, sets inference parameters like `temperature`, `max_new_tokens`, and specifies `test_100.jsonl` as the validation dataset. The results are saved to `results_checkpoint939.jsonl`.

In [ ]:
!pip install jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.8 MB/s eta 0:00:00


### Install `jiwer`
This cell installs the `jiwer` library, which provides robust implementations for calculating evaluation metrics commonly used in speech recognition and optical character recognition (OCR) tasks, such as Character Error Rate (CER) and Word Error Rate (WER).

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.7552
WER: 1.0000


### Calculate CER and WER for Checkpoint 939 (100 Samples)
This cell loads the inference results from `results_checkpoint939.jsonl`, extracts the predicted (`pred`) and ground truth (`ref`) text, and then calculates the Character Error Rate (CER) and Word Error Rate (WER) using the `jiwer` library. These metrics quantify the accuracy of the model's predictions.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 24.48%


### Calculate Character Accuracy for Checkpoint 939 (100 Samples)
This cell converts the Character Error Rate (CER) into a more intuitive Character Accuracy percentage (`(1 - overall_cer) * 100`). A higher character accuracy indicates better performance at the individual character level.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 0.00%


### Calculate Exact Match Accuracy for Checkpoint 939 (100 Samples)
This cell calculates the exact match accuracy, which is the percentage of predictions that are identical to their corresponding ground truth labels. This is a very strict metric, and a low exact match accuracy can still accompany reasonable CER/WER if mistakes are minor.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : పోద్పోంలో
--------------------------------------------------
GT   : పదులు
Pred : పోకలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : టోణాకివాలి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : పౌష్యిచిం
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : సంత్కయోంద్
--------------------------------------------------
GT   : కృషికి
Pred : ప్యికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చెంట్టుష్యినీ
--------------------------------------------------
GT   : లేదంటోంది
Pred : భింతోంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : శారాష్షో
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పోట్యాలలల
--------------------------------------------------
GT   : మెరీనా
Pred : మెందీనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : టూటారాన్య
--------------------------------

### Display Incorrect Predictions for Checkpoint 939 (100 Samples)
This cell iterates through the predictions and ground truth labels, printing pairs where the prediction does not exactly match the reference. This provides qualitative insight into the types of errors the model is making and helps in understanding its limitations.

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v0-20260619-045844/checkpoint-939 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_2/v0-20260619-045844/checkpoint-939 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-19 08:37:42.122059: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critic

### Run Inference on 1000-Sample Test Set (Checkpoint 939)
This cell performs another inference run, similar to the previous one, but this time it uses the larger `test_small.jsonl` (1000 samples) to get a more comprehensive evaluation of checkpoint-939. The results are saved to `results_checkpoint939_2.jsonl`.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint939_2.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.7384
WER: 0.9970


### Calculate CER and WER for Checkpoint 939 (1000 Samples)
This cell calculates the Character Error Rate (CER) and Word Error Rate (WER) for checkpoint-939, based on the inference results from the `results_checkpoint939_2.jsonl` file (1000 samples). This provides a more robust performance assessment than the 100-sample evaluation.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 26.16%


### Calculate Character Accuracy for Checkpoint 939 (1000 Samples)
This cell converts the calculated CER for the 1000-sample test set into a character accuracy percentage, offering a clearer understanding of the model's performance at the character level on a larger evaluation set.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 0.30%


### Calculate Exact Match Accuracy for Checkpoint 939 (1000 Samples)
This cell computes the exact match accuracy for checkpoint-939 on the 1000-sample test set. This metric indicates how many predictions perfectly match the ground truth, providing a strict measure of overall correctness.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : పోద్పోంలో
--------------------------------------------------
GT   : పదులు
Pred : పోకలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : టోణాకివాలి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : పౌష్యిచిం
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : సంత్కయోంద్
--------------------------------------------------
GT   : కృషికి
Pred : ప్యికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చెంట్టుష్యినీ
--------------------------------------------------
GT   : లేదంటోంది
Pred : భింతోంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : శారాష్షో
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : పోట్యాలలల
--------------------------------------------------
GT   : మెరీనా
Pred : మెందీనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : టూటారాన్య
--------------------------------

### Display Incorrect Predictions for Checkpoint 939 (1000 Samples)
This cell displays a selection of incorrect predictions from checkpoint-939 on the 1000-sample test set, helping to qualitatively analyze the types of errors and identify areas for improvement. The output is truncated for brevity.

##Evaluation of chcekpoint 1260 which was trained on whole training dataset

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full/v2-20260619-173334/checkpoint-1260 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1260.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full/v2-20260619-173334/checkpoint-1260 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1260.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-20 11:28:40.911795: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-cri

### Run Inference on 1000-Sample Test Set (Checkpoint 1260)
This cell executes an inference run using a different model checkpoint, `checkpoint-1260`, which was trained on the full dataset. It evaluates its performance on the `test_small.jsonl` (1000 samples) and saves the results to `results_checkpoint1260.jsonl`.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1260.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.4953
WER: 0.9570


### Calculate CER and WER for Checkpoint 1260
This cell loads the inference results from `results_checkpoint1260.jsonl` and calculates the Character Error Rate (CER) and Word Error Rate (WER) for checkpoint-1260. These metrics indicate the accuracy of the model trained on the full dataset.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 50.47%


### Calculate Character Accuracy for Checkpoint 1260
This cell converts the CER of checkpoint-1260 into a character accuracy percentage. A higher character accuracy suggests better performance at recognizing individual characters after training on the complete dataset.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 4.30%


### Calculate Exact Match Accuracy for Checkpoint 1260
This cell computes the exact match accuracy for checkpoint-1260. This strict metric indicates the percentage of predictions that are perfectly identical to the ground truth labels, reflecting the model's overall correctness on the full dataset.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : యస్థియంలో
--------------------------------------------------
GT   : పదులు
Pred : పదాలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : లులోకాదాలాచ్చి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రివణండ
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : ఇంతకుయింద్
--------------------------------------------------
GT   : కృషికి
Pred : ట్రషి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడితేన్ని
--------------------------------------------------
GT   : లేదంటోంది
Pred : పెదంతాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారివ్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రత్యాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : మెరిగనా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కులుంభాన్ని
----------------------------

### Display Incorrect Predictions for Checkpoint 1260
This cell displays examples of predictions from checkpoint-1260 that do not match their ground truth labels. This qualitative analysis helps to understand the types of errors made by the model after training on the full dataset.

## Continuing training on whole dataset from checkpoint-1260

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full/v2-20260619-173334/checkpoint-1260 \
    --dataset /content/drive/MyDrive/Train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full/v2-20260619-173334/checkpoint-1260 --dataset /content/drive/MyDrive/Train_full.jsonl --val_dataset /content/drive/MyDrive/Val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.s

### Continue Training from Checkpoint 1260 (Full Dataset)
This cell continues the training of the GOT-OCR2.0 model from `checkpoint-1260` using the `swift sft` command. It now utilizes the full `Train_full.jsonl` and `Val_2k.jsonl` datasets, aiming to further improve the model's performance. The training was interrupted due to `KeyboardInterrupt`.

## Evaluation of checkpoint 1800

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v1-20260620-121006/checkpoint-1800 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 1 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1800.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v1-20260620-121006/checkpoint-1800 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 1 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1800.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-21 16:12:29.328680: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-c

## Evaluation metrics of checkpoint-1800

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint1800.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.4650
WER: 0.9460


In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 53.50%


In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 5.40%


In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : మిస్తియంలో
--------------------------------------------------
GT   : పదులు
Pred : పదాలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : టీశ్యాకాకాలిచ్చి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రివణంద్ర
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : ఇంతక్యముంద్
--------------------------------------------------
GT   : కృషికి
Pred : ట్రషికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడితేవన్నది
--------------------------------------------------
GT   : లేదంటోంది
Pred : వెదంటాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారవాన్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రతిక్యాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : మెరిగానా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కుటుంధాన్ని
--------------

## Evaluation of checkpoint-5220

In [ ]:
!swift infer \
    --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-5220 \
    --infer_backend pt \
    --temperature 0 \
    --max_new_tokens 512 \
    --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl \
    --max_batch_size 8 \
    --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint5220.jsonl

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v12-20260623-060835/checkpoint-5220 --infer_backend pt --temperature 0 --max_new_tokens 512 --val_dataset /content/drive/MyDrive/Telugu_GOT_OCR/jsons/test_small.jsonl --max_batch_size 8 --result_path /content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint5220.jsonl`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-23 09:12:49.303924: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-

In [ ]:
!pip install jiwer -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.4 MB/s eta 0:00:00


In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint5220.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.3699
WER: 0.8850


In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 63.01%


In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 11.50%


In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : యిస్తియంలో
--------------------------------------------------
GT   : పదులు
Pred : పదలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : అలోకాదాలిస్టీ
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రివచంద్ర
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : ఇంతకుయుంద్
--------------------------------------------------
GT   : కృషికి
Pred : ప్రషికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడివ్విక్రి
--------------------------------------------------
GT   : లేదంటోంది
Pred : వేదంటోంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారవాన్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రతాలతలు
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కుటింభాన్ని
--------------------------------------------------
GT   : తర్వాతయినా
Pred : ఆధతయినా
--------------------

## Continuing Training from checkpoint-5220

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v17-20260623-093450/checkpoint-6660 \
    --dataset /content/drive/MyDrive/Train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v17-20260623-093450/checkpoint-6660 --dataset /content/drive/MyDrive/Train_full.jsonl --val_dataset /content/drive/MyDrive/Val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gn

## Continuing training on whole dataset from checkpoint-7740

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v13-20260624-043716/checkpoint-7740 \
    --dataset /content/drive/MyDrive/Train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v13-20260624-043716/checkpoint-7740 --dataset /content/drive/MyDrive/Train_full.jsonl --val_dataset /content/drive/MyDrive/Val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/p

### Continue Training from Checkpoint 7740 (Full Dataset)
This cell resumes training from `checkpoint-7740`, continuing the process on the full `Train_full.jsonl` and `Val_2k.jsonl` datasets. This iterative training approach allows for longer training sessions and potentially better model convergence. The training was interrupted due to `KeyboardInterrupt`.

## Continuing the training on whole training dataset from checkpoint-11700

In [ ]:
!swift sft \
    --model stepfun-ai/GOT-OCR2_0 \
    --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v22-20260625-065910/checkpoint-11700 \
    --dataset /content/drive/MyDrive/Train_full.jsonl \
    --val_dataset /content/drive/MyDrive/Val_2k.jsonl \
    --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 \
    --torch_dtype float16 \
    --fp16 true \
    --gradient_checkpointing false \
    --num_train_epochs 3 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --learning_rate 5e-5 \
    --max_length 512 \
    --eval_strategy steps \
    --eval_steps 150 \
    --save_strategy steps \
    --save_steps 150 \
    --save_total_limit 5 \
    --load_best_model_at_end true \
    --create_checkpoint_symlink true \
    --early_stop_interval 5

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model stepfun-ai/GOT-OCR2_0 --resume_from_checkpoint /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2/v22-20260625-065910/checkpoint-11700 --dataset /content/drive/MyDrive/Train_full.jsonl --val_dataset /content/drive/MyDrive/Val_2k.jsonl --output_dir /content/drive/MyDrive/Telugu_GOT_OCR/output_weights_full_2 --torch_dtype float16 --fp16 true --gradient_checkpointing false --num_train_epochs 3 --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --learning_rate 5e-5 --max_length 512 --eval_strategy steps --eval_steps 150 --save_strategy steps --save_steps 150 --save_total_limit 5 --load_best_model_at_end true --create_checkpoint_symlink true --early_stop_interval 5`
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-g

### Continue Training from Checkpoint 11700 (Full Dataset)
This cell continues the training process from `checkpoint-11700`, again using the full training and validation datasets. This demonstrates a strategy of picking up training from the latest checkpoint to optimize the model over extended periods. The training was interrupted due to `KeyboardInterrupt`.

## Evaluating on checkpoint-7200 on 9000 Images


In [ ]:
import json

jsonl_file = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint_full_7200.jsonl"

count = 0

with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():  # Ignore empty lines
            json.loads(line)  # Optional: verify it's valid JSON
            count += 1

print(f"Total predictions: {count}")

Total predictions: 9000


### Count Predictions in Result File
This cell counts the number of lines (and thus predictions) in the `results_checkpoint_full_7200.jsonl` file. This verifies the completeness of the inference output for `checkpoint-7200`.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint_full_7200.jsonl"

predictions = []
references = []

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)

        pred = data["response"].strip()
        ref = data["labels"].strip()

        predictions.append(pred)
        references.append(ref)

overall_cer = cer(references, predictions)
overall_wer = wer(references, predictions)

print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}")

CER: 0.3306
WER: 0.8657


### Calculate CER and WER for Checkpoint 7200
This cell calculates the Character Error Rate (CER) and Word Error Rate (WER) for `checkpoint-7200` based on the `results_checkpoint_full_7200.jsonl` file. These metrics provide a quantitative assessment of the model's accuracy at this particular training stage.

In [ ]:
char_accuracy = (1 - overall_cer) * 100

print(f"Character Accuracy: {char_accuracy:.2f}%")

Character Accuracy: 66.94%


### Calculate Character Accuracy for Checkpoint 7200
This cell converts the CER of `checkpoint-7200` into a character accuracy percentage, offering a clear measure of how well the model performs at recognizing individual characters after continued training.

In [ ]:
correct = sum(
    pred == ref
    for pred, ref in zip(predictions, references)
)

accuracy = correct / len(predictions)

print(f"Exact Match Accuracy: {accuracy*100:.2f}%")

Exact Match Accuracy: 13.43%


### Calculate Exact Match Accuracy for Checkpoint 7200
This cell computes the exact match accuracy for `checkpoint-7200`. This metric indicates the percentage of predictions that are perfectly identical to the ground truth, providing a strict measure of overall correctness for this checkpoint.

In [ ]:
for pred, ref in zip(predictions, references):
    if pred != ref:
        print("GT   :", ref)
        print("Pred :", pred)
        print("-"*50)

GT   : మస్తిష్కంలో
Pred : మసిష్కంలో
--------------------------------------------------
GT   : పదులు
Pred : పదలు
--------------------------------------------------
GT   : తిలోకాదకాలిచ్చి
Pred : టలోకాదకాలచ్చి
--------------------------------------------------
GT   : హేమచంద్ర
Pred : ప్రివచంద్ర
--------------------------------------------------
GT   : ఇంతకుముందు
Pred : ఇంతకుయుదు
--------------------------------------------------
GT   : కృషికి
Pred : టీషికి
--------------------------------------------------
GT   : చూడలేమన్నది
Pred : చూడివన్నది
--------------------------------------------------
GT   : లేదంటోంది
Pred : వేదంటాంది
--------------------------------------------------
GT   : గౌరవాన్ని
Pred : గారవాన్ని
--------------------------------------------------
GT   : ప్రతికూలతలు
Pred : ప్రతకాలతలు
--------------------------------------------------
GT   : మెరీనా
Pred : మెరిగా
--------------------------------------------------
GT   : కుటుంబాన్ని
Pred : కుటించాన్ని
-----------------------------

### Display Incorrect Predictions for Checkpoint 7200
This cell displays examples of incorrect predictions from `checkpoint-7200`, showing pairs of ground truth and predicted text that do not match. This helps in qualitatively analyzing the remaining errors and understanding the model's persistent challenges.

In [ ]:
import json
from jiwer import cer, wer
from collections import Counter

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint_full_7200.jsonl"  # update path as needed

records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        pred = data["response"].strip()
        ref = data["labels"].strip()
        try:
            c = cer([ref], [pred])
        except Exception:
            c = 1.0
        records.append({"pred": pred, "ref": ref, "cer": c})

# Sort by worst CER
records.sort(key=lambda x: -x["cer"])

print("=== TOP 15 WORST PREDICTIONS ===\n")
for r in records[:15]:
    print(f"CER: {r['cer']:.2f}")
    print(f"REF : {r['ref']}")
    print(f"PRED: {r['pred']}")
    print(f"Ref len: {len(r['ref'])}, Pred len: {len(r['pred'])}")
    print("-" * 60)

# Check for repetition patterns (degenerate generation)
def has_repetition(s, min_repeat=4):
    for i in range(len(s) - min_repeat):
        chunk = s[i:i+2]
        if chunk * min_repeat in s:
            return True
    return False

repetitive = [r for r in records if has_repetition(r["pred"])]
print(f"\n=== PREDICTIONS WITH REPETITION PATTERNS: {len(repetitive)} / {len(records)} ===")

# Length mismatch analysis
big_mismatch = [r for r in records if abs(len(r["pred"]) - len(r["ref"])) > 0.5 * len(r["ref"])]
print(f"=== PREDICTIONS WITH >50% LENGTH MISMATCH: {len(big_mismatch)} / {len(records)} ===")

# Character-level confusion (most common substituted/missing chars in worst 100)
ref_chars = Counter()
pred_chars = Counter()
for r in records[:100]:
    ref_chars.update(r["ref"])
    pred_chars.update(r["pred"])

print("\n=== TOP 15 CHARS IN REFERENCE (worst 100 samples) ===")
for ch, cnt in ref_chars.most_common(15):
    print(f"{repr(ch)}: {cnt}")

=== TOP 15 WORST PREDICTIONS ===

CER: 37.44
REF : మాధ్యమంతో
PRED: చేస్తించునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునునును
Ref len: 9, Pred len: 339
------------------------------------------------------------
CER: 7.53
REF : రక్షించుకోవడానికి
PRED: చెట్లులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులులు
Ref len: 17, Pred len: 130
------------------------------------------------------------
CER: 4.80
REF : గాడిన
PRED: చెట్లులులులులులులులులులు
Ref len: 5, Pred len: 24
------------------------------------------------------------
CER: 4.25
REF : రాను
PRED: చ్చించ్చేస్తున్నారు
Ref len: 4, Pred len: 19
------------------------------------

In [ ]:
!pip install Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.5 MB/s eta 0:00:00


### Install `Levenshtein`
This cell installs the `Levenshtein` library, which provides functions for calculating the Levenshtein distance (edit distance) between two strings. This metric is fundamental for detailed error analysis in OCR and sequence prediction tasks.

In [ ]:
import json
from jiwer import cer, wer

path = "/content/drive/MyDrive/Telugu_GOT_OCR/results_checkpoint_full_7200.jsonl"  # update path as needed

records = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        pred = data["response"].strip()
        ref = data["labels"].strip()
        records.append({"pred": pred, "ref": ref})

all_refs = [r["ref"] for r in records]
all_preds = [r["pred"] for r in records]

overall_cer = cer(all_refs, all_preds)
overall_wer = wer(all_refs, all_preds)
print(f"=== FULL SET (n={len(records)}) ===")
print(f"CER: {overall_cer:.4f}")
print(f"WER: {overall_wer:.4f}\n")

# Identify outliers: prediction length > 3x reference length (or ref is very short and pred is long)
def is_outlier(r, ratio=3.0, min_ref_len=1):
    ref_len = max(len(r["ref"]), min_ref_len)
    return len(r["pred"]) > ratio * ref_len

outliers = [r for r in records if is_outlier(r)]
clean = [r for r in records if not is_outlier(r)]

print(f"=== OUTLIERS (pred > 3x ref length): {len(outliers)} / {len(records)} ({100*len(outliers)/len(records):.2f}%) ===\n")

if clean:
    clean_refs = [r["ref"] for r in clean]
    clean_preds = [r["pred"] for r in clean]
    clean_cer = cer(clean_refs, clean_preds)
    clean_wer = wer(clean_refs, clean_preds)
    print(f"=== EXCLUDING OUTLIERS (n={len(clean)}) ===")
    print(f"CER: {clean_cer:.4f}")
    print(f"WER: {clean_wer:.4f}\n")

# Reference length distribution for outliers vs clean
import statistics
if outliers:
    outlier_ref_lens = [len(r["ref"]) for r in outliers]
    print(f"Outlier ref lengths -> mean: {statistics.mean(outlier_ref_lens):.1f}, "
          f"median: {statistics.median(outlier_ref_lens)}, "
          f"max: {max(outlier_ref_lens)}")

clean_ref_lens = [len(r["ref"]) for r in clean]
print(f"Clean ref lengths    -> mean: {statistics.mean(clean_ref_lens):.1f}, "
      f"median: {statistics.median(clean_ref_lens)}, "
      f"max: {max(clean_ref_lens)}")

# How much do outliers contribute to total edit distance?
import Levenshtein as Lev  # pip install python-Levenshtein if not available
total_edits = sum(Lev.distance(r["ref"], r["pred"]) for r in records)
outlier_edits = sum(Lev.distance(r["ref"], r["pred"]) for r in outliers)
print(f"\nOutliers contribute {outlier_edits}/{total_edits} "
      f"({100*outlier_edits/total_edits:.1f}%) of total edit distance, "
      f"despite being only {100*len(outliers)/len(records):.1f}% of samples")

=== FULL SET (n=9000) ===
CER: 0.3306
WER: 0.8657

=== OUTLIERS (pred > 3x ref length): 9 / 9000 (0.10%) ===

=== EXCLUDING OUTLIERS (n=8991) ===
CER: 0.3239
WER: 0.8655

Outlier ref lengths -> mean: 5.1, median: 4, max: 17
Clean ref lengths    -> mean: 8.7, median: 8, max: 26

Outliers contribute 541/25896 (2.1%) of total edit distance, despite being only 0.1% of samples


### Detailed Error Analysis with Outlier Detection
This cell performs a comprehensive error analysis on the inference results from `checkpoint-7200`. It calculates overall CER and WER, identifies and analyzes "outlier" predictions (where prediction length is significantly different from reference length), and quantifies their contribution to the total edit distance. This helps in understanding the nature of the model's errors and potential data issues.